In [1]:
import modules.data_scvi as d 
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device()#drop=['cuda:4','cuda:7'])

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78941)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 60259)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 38933)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 28839)

# #### Device() ####
# device = cuda:4



---

In [2]:
from modules.layers import AttentionSetPooling
from modules.loss import MultiLoss, NBLoss, KLDLoss
from modules.model import CountsAutoencoder, NBGLM
from modules.norm import LogCounts
from modules.train import Experiment, grid
from modules.trainers import ReconstrTrainer

from functools import partial
import torch.nn as nn

In [3]:
#### data ####
datasets = {
    'cortex': d.scviData(
        dataset='cortex', # cortex, pbmc_dataset
        dataset_dir=dataset_dir / 'scvi',
        x_label_col='index',
        x_label_type='name', # name, ensg
        y_label_col='cell_type',
        gene_name_path=dataset_dir/'other'/'name2ensg.csv',
    ),

    'pbmc_dataset': d.scviData(
        dataset='pbmc_dataset', # cortex, pbmc_dataset
        dataset_dir=dataset_dir / 'scvi',
        x_label_col='index',
        x_label_type='ensg', # name, ensg
        y_label_col='str_labels', # cell_type, str_labels
        gene_name_path=dataset_dir/'other'/'name2ensg.csv',
    ),
}

INFO     File /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/cortex/expression.bin already downloaded   
INFO     Loading Cortex data from /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/cortex/expression.bin  
INFO     Finished loading Cortex data                                                                              


/home/mv18gs/miniconda3/envs/thesis_scvi/lib/python3.14/functools.py:982: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


INFO     File /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/pbmc_dataset/gene_info_pbmc.csv already    
         downloaded                                                                                                
INFO     File /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/pbmc_dataset/pbmc_metadata.pickle already  
         downloaded                                                                                                
INFO     File                                                                                                      
         /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/pbmc_dataset/pbmc8k/filtered_gene_bc_matrices.ta
         r.gz already downloaded                                                                                   
INFO     Extracting tar file                                                                                       
INFO     Removing extracted data at                                     

/home/mv18gs/miniconda3/envs/thesis_scvi/lib/python3.14/site-packages/scvi/data/_built_in_data/_pbmc.py:75: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = pbmc8k.concatenate(pbmc4k)


In [4]:
def run_expt(counts_name:str, counts_data:d.TCGA, num_trials:int=5):
    # data
    kegg = d.KEGG(
        relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
        counts_data=counts_data,
    )
    dataset = d.GraphDataset(counts_data, kegg, kegg)

    # models
    glm = NBGLM(dataset)
    vae = CountsAutoencoder(
        dataset=dataset,
        embed_dim = 128,
        method='node',


        norm_class = LogCounts,
        encoder_class = nn.Linear,
        pooling_class = AttentionSetPooling,
        mlp = False,
        variational = True,

        hidden_dims = 1,
        act_fn = nn.ReLU, 
        norm_fn = 'layer', 
        end_fn = False,

        norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True,}
    )
    pw = CountsAutoencoder(
        dataset=dataset,
        embed_dim = 128,
        method='set',


        norm_class = LogCounts,
        encoder_class = nn.Linear,
        pooling_class = AttentionSetPooling,
        mlp = False,
        variational = True,

        hidden_dims = 1,
        act_fn = nn.ReLU, 
        norm_fn = 'layer', 
        end_fn = False,

        norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True,}
    )

    # trainer
    glm_trainer = ReconstrTrainer(
        lr=1e-3, 

        trainer_norm_class=LogCounts,
        early_stop=True,
        stop_metric='mae',

        kw_keys={'x':'x', 'mu':'x_pred', 'theta':'theta'},
        metric_keys={'pred':'mu', 'target':'x'},
        loss_class=NBLoss,
    )

    vae_trainer = ReconstrTrainer(
        lr=1e-3, 

        trainer_norm_class=LogCounts,
        early_stop=True,
        stop_metric='mae',

        kw_keys=('x','x_pred','theta','z_mu','z_logvar'),
        metric_keys={'pred':'x_pred', 'target':'x'},
        
        loss_class=MultiLoss,
        loss_kwargs = {
        'loss_classes': [NBLoss, KLDLoss],
        'loss_weights': [1.0, 1e-4],
        'loss_inputs': [
            {'x':'x', 'mu':'x_pred', 'theta':'theta'},
            {'mu':'z_mu', 'logvar':'z_logvar'}
        ]
    },
    )
    
    # expt
    expt = Experiment(
        num_trials=num_trials, # scVI: 5 trials
        num_epochs=300, # scVI: 200 epochs
        dataset=dataset,
        device=device,
        batch_size=128
    )

    expt.add_config(
        name='glm',
        model=glm,
        trainer=glm_trainer,
    )

    expt.add_config(
        name='gene-vae',
        model=vae,
        trainer=vae_trainer,
    )

    expt.add_config(
        name='pathway-vae',
        model=pw,
        trainer=vae_trainer,
    )

    expt.run_experiment(
        f'benchmark_1b_nbreconstr_scvi_{counts_name}',
        report_metrics=['loss','kld','nb','mae', 'accuracy', 'precision', 'f1'],
        # save_model=True,
        save_csv=True,
        save_params=True,
        save_values=True,
        verbose=False,
    )

In [5]:
for k,v in datasets.items():
    run_expt(k, v, num_trials=25)
``
# for k,v in datasets.items():
#     run_expt(k, v, num_trials=25)


# for k,v in datasets.items():
#     run_expt(k, v, num_trials=20)

# for some reason running again causes a bug, like re-initializing the dataset is needed

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     3688                     list
# pathway_labels           305                      list
# edge_index               (2, 26946)               Tensor (cuda:4)
# edge_attr                (26946, 16)              Tensor (cuda:4)
# edge_labels              16                       list
# pathway_index            (3688, 305)              Tensor (cuda:4)

# #### scviData() ####
# _orig_kwargs             8                        dict
# verbose                  True                     bool
# adata                    AnnData
# x_names                  (3688, 3)                DataFrame
# ensg_complete            14404                    list
# x_labels                 19972                    list
# y                        (3005,)                  Tensor (cuda:4)
# y_labels                 7                        list
# x          

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

...

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

/home/mv18gs/Documents/GitHub/pathway_model/projects/2_refactored_exp/modules/train.py:840: SmallSampleWarning: After omitting NaNs, one or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  sem = stats.sem(series, nan_policy='omit')
